# Yemen MarketShield AI — WFP HAPI temporal validation

This notebook reproduces the external-data audit and time-split experiment described in `MODEL_VALIDATION_REPORT.md`.

**Important:** This is research validation only. The World Bank Real-Time Prices data may share provenance with WFP; this is a later-period test, not proof of independent-source validation. Do not use the results for operational humanitarian, financial, or procurement decisions.

## Files to upload
Provide both input files locally or upload them in Colab:
1. `WFP_Wheat_Yemen_HAPI_Response.json` — a saved response snapshot from HDX HAPI, queried for Yemen, wheat commodity code `84`, actual retail prices, currency YER, unit KG, through October 2026. This raw input is not contained in `Yemen_MarketShield_WFP_Validation_Outputs.zip` and is intentionally not committed to the public GitHub repository.
2. `YEM_RTFP_mkt_2009_2026-03-24.zip` — the original World Bank food-price archive. Raw source snapshots are not committed to public GitHub; see the README for official catalogue links.

The original HAPI result was queried for commodity code `84` (Wheat), `YEM`, and period through October 2026. This notebook does not need the exchange-rate or energy ZIPs because the current test uses only wheat prices.

In [1]:
import json, zipfile, os
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

try:
    from google.colab import files
    uploaded = files.upload()
    uploaded_names = list(uploaded.keys())
    print("Colab uploaded:", uploaded_names)
except ImportError:
    files = None
    uploaded_names = [str(p) for p in Path(".").iterdir()
                      if p.is_file() and p.suffix.lower() in {".json", ".txt", ".zip"}]
    print("Local source files discovered:", uploaded_names)

# Locate the HAPI response by file contents, so filenames need not be exact.
hapi_name = None
hapi_obj = None
for name in uploaded_names:
    if name.lower().endswith((".json", ".txt")):
        try:
            with open(name, "r", encoding="utf-8-sig") as f:
                candidate = json.load(f)
            if isinstance(candidate, dict) and isinstance(candidate.get("data"), list):
                first = candidate["data"][0] if candidate["data"] else {}
                if "commodity_code" in first and "reference_period_start" in first:
                    hapi_name, hapi_obj = name, candidate
                    break
        except (json.JSONDecodeError, UnicodeDecodeError):
            pass

wb_zip_name = next(
    (name for name in uploaded_names
     if name.lower().endswith(".zip") and "rtfp" in name.lower()),
    None
)
if hapi_obj is None:
    raise ValueError("Provide WFP_Wheat_Yemen_HAPI_Response.json, or upload the HAPI response in Colab.")
if wb_zip_name is None:
    raise ValueError("Provide YEM_RTFP_mkt_2009_2026-03-24.zip, or upload it in Colab.")

hapi = pd.DataFrame(hapi_obj["data"])
with zipfile.ZipFile(wb_zip_name) as z:
    csv_name = next(
        n for n in z.namelist()
        if n.lower().endswith(".csv") and "rtfp" in n.lower()
    )
    wb_food = pd.read_csv(z.open(csv_name), low_memory=False)

print("HAPI response file:", hapi_name)
print("HAPI rows:", len(hapi), "| World Bank food file shape:", wb_food.shape)


Local source files discovered: ['YEM_RTEP_mkt_2009_2026-03-24.zip', 'requirements-validation.txt', 'Yemen_MarketShield_WFP_Validation_Outputs.zip', 'YEM_RTFX_mkt_2009_2026-03-24.zip', 'requirements.txt', 'WFP_Wheat_Yemen_HAPI_Response.json', 'YEM_RTFP_mkt_2009_2026-03-24.zip']


HAPI response file: WFP_Wheat_Yemen_HAPI_Response.json
HAPI rows: 936 | World Bank food file shape: (5175, 165)


In [2]:
# Restrict to the wheat series used in the requested comparison.
hapi["commodity_code"] = hapi["commodity_code"].astype(str)
hapi["market_code"] = hapi["market_code"].astype(str)
hapi["reference_period_start"] = pd.to_datetime(hapi["reference_period_start"])
hapi["reference_period_end"] = pd.to_datetime(hapi["reference_period_end"])
hapi["price"] = pd.to_numeric(hapi["price"], errors="coerce")

hapi = hapi[
    hapi["commodity_code"].eq("84")
    & hapi["commodity_name"].str.casefold().eq("wheat")
    & hapi["price_flag"].eq("actual")
    & hapi["price_type"].eq("Retail")
    & hapi["currency_code"].eq("YER")
    & hapi["unit"].eq("KG")
].copy()

hapi["period_month"] = hapi["reference_period_start"].dt.to_period("M").dt.to_timestamp()
hapi = hapi.sort_values(["market_code", "reference_period_start"]).reset_index(drop=True)

audit = {
    "rows": len(hapi),
    "markets": hapi["market_name"].nunique(),
    "start_period": str(hapi["reference_period_start"].min()),
    "end_period_start": str(hapi["reference_period_start"].max()),
    "end_period_end": str(hapi["reference_period_end"].max()),
    "price_flags": sorted(hapi["price_flag"].dropna().unique().tolist()),
    "price_types": sorted(hapi["price_type"].dropna().unique().tolist()),
    "currencies": sorted(hapi["currency_code"].dropna().unique().tolist()),
    "units": sorted(hapi["unit"].dropna().unique().tolist()),
    "duplicate_market_period_keys": int(hapi.duplicated(
        ["market_code", "commodity_code", "reference_period_start"]
    ).sum()),
}
display(pd.DataFrame([audit]).T.rename(columns={0: "Value"}))
display(hapi.groupby("market_name").size().sort_values(ascending=False).rename("rows").to_frame())

,Value
rows,936
markets,23
start_period,2021-12-15 00:00:00
end_period_start,2026-09-15 00:00:00
end_period_end,2026-10-14 23:59:59
price_flags,[actual]
price_types,[Retail]
currencies,[YER]
units,[KG]
duplicate_market_period_keys,0


,rows
market_name,
Aden City,57
Al Hawtah- Main road - Back Street,57
Al Ghaidha,57
Zungubar City,57
Mukalla City,57
Sayoun City,57
Central Ataq - Al Nasib neighbour - Darhim,56
Soqatra (Hudaibo),55
Addaleh Town - Al Tadamhn,53


## Historical agreement with World Bank `c_wheat`

The matching rule below aligns the HAPI monthly record to the calendar month of `reference_period_start`. HAPI periods cover the 15th to the 14th, so this month alignment is approximate, not an exact day-for-day match. The World Bank `c_wheat` column is an estimate field, while HAPI records selected here are marked `actual`.

In [3]:
wb_food["price_date"] = pd.to_datetime(wb_food["price_date"])
wb_small = wb_food[
    ["price_date", "mkt_name", "c_wheat", "wheat", "currency"]
].copy()

pairs = hapi.merge(
    wb_small,
    left_on=["market_name", "period_month"],
    right_on=["mkt_name", "price_date"],
    how="left",
    validate="one_to_one",
    indicator=True,
)
hist = pairs[
    pairs["_merge"].eq("both")
    & pairs["period_month"].between(
        pd.Timestamp("2022-01-01"), pd.Timestamp("2026-03-01")
    )
].copy()
hist["abs_error"] = (hist["price"] - hist["c_wheat"]).abs()
hist["sq_error"] = (hist["price"] - hist["c_wheat"]) ** 2

hist_metrics = pd.DataFrame([{
    "Matched pairs": len(hist),
    "Markets": hist["market_name"].nunique(),
    "Period start": hist["period_month"].min().strftime("%Y-%m"),
    "Period end": hist["period_month"].max().strftime("%Y-%m"),
    "MAE (YER/kg)": hist["abs_error"].mean(),
    "RMSE (YER/kg)": np.sqrt(hist["sq_error"].mean()),
    "WAPE (%)": 100 * hist["abs_error"].sum() / hist["price"].abs().sum(),
    "Pearson correlation": hist[["price", "c_wheat"]].corr().iloc[0, 1],
    "Mean bias (HAPI - WB)": (hist["price"] - hist["c_wheat"]).mean(),
}])
display(hist_metrics.round(4))
hist.to_csv(results_dir / "wfp_worldbank_historical_pairs.csv", index=False, lineterminator="\n")

,Matched pairs,Markets,Period start,Period end,MAE (YER/kg),RMSE (YER/kg),WAPE (%),Pearson correlation,Mean bias (HAPI - WB)
0,851,23,2022-01,2026-03,18.2002,46.2903,2.7053,0.9879,2.6541


## Build calendar-aligned lag features

Lags are joined by market and exact calendar month, not by row position. This avoids treating a value from two or three months earlier as if it were the previous month when an observation is missing.

In [4]:
wf = hapi.copy().sort_values(["market_code", "reference_period_start"]).reset_index(drop=True)

for lag in [1, 2, 3, 6, 12]:
    lookup = wf[["market_code", "period_month", "price"]].copy()
    lookup["period_month"] = lookup["period_month"] + pd.DateOffset(months=lag)
    lookup = lookup.rename(columns={"price": f"lag_price_{lag}"})
    wf = wf.merge(
        lookup,
        on=["market_code", "period_month"],
        how="left",
        validate="one_to_one",
    )

wf["pct_change_previous"] = np.where(
    wf["lag_price_1"].notna()
    & wf["lag_price_2"].notna()
    & wf["lag_price_2"].ne(0),
    (wf["lag_price_1"] - wf["lag_price_2"]) / wf["lag_price_2"],
    np.nan,
)
wf["month"] = wf["period_month"].dt.month
wf["month_sin"] = np.sin(2 * np.pi * wf["month"] / 12)
wf["month_cos"] = np.cos(2 * np.pi * wf["month"] / 12)
wf["target_rise5"] = np.nan
has_previous = wf["lag_price_1"].notna()
wf.loc[has_previous, "target_rise5"] = (
    wf.loc[has_previous, "price"]
    > wf.loc[has_previous, "lag_price_1"] * 1.05
).astype(int)

numeric_features = [
    "lag_price_1", "lag_price_2", "lag_price_3",
    "lag_price_6", "lag_price_12",
    "pct_change_previous", "month_sin", "month_cos"
]
categorical_features = ["market_name"]
features = numeric_features + categorical_features

validation_start = pd.Timestamp("2025-04-01")
test_start = pd.Timestamp("2026-04-01")
test_end = pd.Timestamp("2026-10-01")

eligible = wf[wf["lag_price_1"].notna()].copy()
train_reg = eligible[eligible["period_month"] < validation_start].copy()
val_reg = eligible[
    eligible["period_month"].between(validation_start, test_start, inclusive="left")
].copy()
test_reg = eligible[
    eligible["period_month"].between(test_start, test_end, inclusive="left")
].copy()

print("Train rows:", len(train_reg))
print("Validation rows:", len(val_reg))
print("Future test rows with previous-month price:", len(test_reg))
print("Test markets:", test_reg["market_name"].nunique())
print("Test months:", sorted(test_reg["period_month"].dt.strftime("%Y-%m").unique()))

Train rows: 710
Validation rows: 103
Future test rows with previous-month price: 62
Test markets: 11
Test months: ['2026-04', '2026-05', '2026-06', '2026-07', '2026-08', '2026-09']


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    precision_score, recall_score, f1_score,
    balanced_accuracy_score, confusion_matrix,
    roc_auc_score, average_precision_score
)
from sklearn.base import clone

def reg_metrics(y_true, pred):
    y_true = np.asarray(y_true, dtype=float)
    pred = np.asarray(pred, dtype=float)
    ae = np.abs(y_true - pred)
    return {
        "MAE": ae.mean(),
        "RMSE": np.sqrt(np.mean((y_true - pred) ** 2)),
        "WAPE (%)": 100 * ae.sum() / np.abs(y_true).sum(),
    }

preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_features),
    ("market", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
])

model_factories = {
    "Ridge": lambda: Pipeline([("prep", clone(preprocess)), ("model", Ridge(alpha=20.0))]),
    "Random Forest": lambda: Pipeline([
        ("prep", clone(preprocess)),
        ("model", RandomForestRegressor(
            n_estimators=300, min_samples_leaf=5, max_features=0.8,
            random_state=42, n_jobs=-1
        ))
    ]),
    "Extra Trees": lambda: Pipeline([
        ("prep", clone(preprocess)),
        ("model", ExtraTreesRegressor(
            n_estimators=300, min_samples_leaf=5, max_features=0.8,
            random_state=42, n_jobs=-1
        ))
    ]),
    "Gradient Boosting": lambda: Pipeline([
        ("prep", clone(preprocess)),
        ("model", GradientBoostingRegressor(
            n_estimators=180, learning_rate=0.03, max_depth=2,
            loss="huber", random_state=42
        ))
    ]),
}

# Select/compare on validation only. The 12-month seasonal comparator requires a lag12,
# so all listed validation comparisons use this common subset.
val_eval = val_reg[val_reg["lag_price_12"].notna()].copy()
validation_rows = [
    {"Model": "Previous period baseline", **reg_metrics(val_eval["price"], val_eval["lag_price_1"])},
    {"Model": "Seasonal naive (12 months)", **reg_metrics(val_eval["price"], val_eval["lag_price_12"])},
]
for name, factory in model_factories.items():
    model = factory()
    model.fit(train_reg[features], train_reg["price"])
    prediction = model.predict(val_eval[features])
    validation_rows.append({"Model": name, **reg_metrics(val_eval["price"], prediction)})

validation_results = pd.DataFrame(validation_rows).sort_values("MAE")
display(validation_results.round(3))
validation_results.to_csv(results_dir / "wfp_regression_validation_metrics.csv", index=False, lineterminator="\n")
print("Best by validation MAE:", validation_results.iloc[0]["Model"])

,Model,MAE,RMSE,WAPE (%)
0,Previous period baseline,80.970,133.396,8.445
5,Gradient Boosting,85.266,129.894,8.893
3,Random Forest,85.827,128.140,8.951
2,Ridge,92.578,136.927,9.655
4,Extra Trees,99.993,134.287,10.429
1,Seasonal naive (12 months),185.182,216.043,19.313


Best by validation MAE: Previous period baseline


## Evaluate on the later WFP periods

The test period is April–September 2026, after the last calendar month available in the supplied World Bank food-price archive. This is a **temporal holdout**, but it is not fully independent by source because World Bank Real-Time Prices may incorporate data from WFP and other sources.

For each test row, the previous WFP period's observed price is available as the one-period baseline. The machine-learning models are fit only on records dated before the test period; test features use only earlier periods for that market.

In [6]:
pretest_train = eligible[eligible["period_month"] < test_start].copy()
test_predictions = test_reg[
    ["market_name", "market_code", "period_month",
     "reference_period_start", "reference_period_end", "price",
     "lag_price_1", "lag_price_12"]
].copy().rename(columns={
    "price": "actual_wfp_price_yer_kg",
    "lag_price_1": "prediction_previous_period",
    "lag_price_12": "prediction_seasonal_12m",
})

test_metrics_rows = [
    {"Model": "Previous period baseline",
     **reg_metrics(test_reg["price"], test_reg["lag_price_1"])},
    {"Model": "Seasonal naive (12 months)",
     **reg_metrics(test_reg["price"], test_reg["lag_price_12"])},
]
for name in ["Gradient Boosting", "Random Forest"]:
    model = model_factories[name]()
    model.fit(pretest_train[features], pretest_train["price"])
    prediction = model.predict(test_reg[features])
    test_predictions[f"prediction_{name.lower().replace(' ', '_')}"] = prediction
    test_metrics_rows.append({"Model": name, **reg_metrics(test_reg["price"], prediction)})

test_results = pd.DataFrame(test_metrics_rows).sort_values("MAE")
test_results.insert(0, "Test period", "2026-04 to 2026-09")
test_results.insert(1, "Rows", len(test_reg))
test_results.insert(2, "Markets", test_reg["market_name"].nunique())
display(test_results.round(3))
test_results.to_csv(results_dir / "wfp_regression_future_test_metrics.csv", index=False, lineterminator="\n")
test_predictions.to_csv(results_dir / "wfp_regression_future_test_predictions.csv", index=False, lineterminator="\n")

,Test period,Rows,Markets,Model,MAE,RMSE,WAPE (%)
0,2026-04 to 2026-09,62,11,Previous period baseline,19.999,33.941,2.409
2,2026-04 to 2026-09,62,11,Gradient Boosting,24.678,32.578,2.973
3,2026-04 to 2026-09,62,11,Random Forest,28.279,38.899,3.407
1,2026-04 to 2026-09,62,11,Seasonal naive (12 months),217.427,269.283,26.193


## Early-warning classifier: >5% rise in the next WFP period

The label is a price more than 5% above the immediately previous calendar month's actual price. The model is fit on the training period, the alert probability threshold is selected on the validation period, and the same fitted classifier/threshold are then evaluated on the future test period. No test data is used to pick the threshold.

In [7]:
def class_metrics(y_true, pred, prob):
    y_true = np.asarray(y_true, dtype=int)
    pred = np.asarray(pred, dtype=int)
    prob = np.asarray(prob, dtype=float)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "rows": len(y_true),
        "positive_rate": y_true.mean(),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, prob) if len(np.unique(y_true)) > 1 else np.nan,
        "average_precision": average_precision_score(y_true, prob) if len(np.unique(y_true)) > 1 else np.nan,
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
    }

class_train = wf[
    (wf["period_month"] < validation_start)
    & wf["lag_price_1"].notna()
    & wf["target_rise5"].notna()
].copy()
class_val = wf[
    (wf["period_month"] >= validation_start)
    & (wf["period_month"] < test_start)
    & wf["lag_price_1"].notna()
    & wf["target_rise5"].notna()
].copy()
class_test = wf[
    (wf["period_month"] >= test_start)
    & (wf["period_month"] < test_end)
    & wf["lag_price_1"].notna()
    & wf["target_rise5"].notna()
].copy()

classifier = Pipeline([
    ("prep", clone(preprocess)),
    ("model", LogisticRegression(
        max_iter=2000, class_weight="balanced", C=0.3, random_state=42
    )),
])
classifier.fit(class_train[features], class_train["target_rise5"].astype(int))
val_prob = classifier.predict_proba(class_val[features])[:, 1]

threshold_rows = []
for threshold in np.linspace(0.05, 0.95, 91):
    pred = (val_prob >= threshold).astype(int)
    m = class_metrics(class_val["target_rise5"].astype(int), pred, val_prob)
    threshold_rows.append({"threshold": float(threshold), **m})

threshold_table = pd.DataFrame(threshold_rows).sort_values(
    ["f1", "precision"], ascending=False
)
selected_threshold = float(threshold_table.iloc[0]["threshold"])
display(threshold_table.head(10).round(3))
print("Validation-selected threshold:", selected_threshold)

test_prob = classifier.predict_proba(class_test[features])[:, 1]
test_pred = (test_prob >= selected_threshold).astype(int)
classifier_test_metrics = class_metrics(
    class_test["target_rise5"].astype(int), test_pred, test_prob
)
classifier_test_metrics["threshold"] = selected_threshold
display(pd.DataFrame([classifier_test_metrics]).round(3))

classifier_predictions = class_test[
    ["market_name", "market_code", "period_month",
     "reference_period_start", "reference_period_end", "price",
     "lag_price_1", "target_rise5"]
].copy().rename(columns={
    "price": "actual_wfp_price_yer_kg",
    "lag_price_1": "previous_period_price",
    "target_rise5": "actual_rise_gt5",
})
classifier_predictions["price_change_pct"] = (
    classifier_predictions["actual_wfp_price_yer_kg"]
    / classifier_predictions["previous_period_price"] - 1
) * 100
classifier_predictions["probability_rise_gt5"] = test_prob
classifier_predictions["predicted_rise_gt5"] = test_pred
threshold_table.to_csv(results_dir / "wfp_classifier_validation_thresholds.csv", index=False, lineterminator="\n")
classifier_predictions.to_csv(results_dir / "wfp_classifier_future_test_predictions.csv", index=False, lineterminator="\n")
pd.DataFrame([classifier_test_metrics]).to_csv(results_dir / "wfp_classifier_future_test_metrics.csv", index=False, lineterminator="\n")

,threshold,rows,positive_rate,balanced_accuracy,precision,recall,f1,roc_auc,average_precision,TN,FP,FN,TP
44,0.49,103,0.252,0.565,0.280,1.000,0.437,0.583,0.344,10,67,0,26
45,0.50,103,0.252,0.565,0.280,1.000,0.437,0.583,0.344,10,67,0,26
56,0.61,103,0.252,0.579,0.293,0.846,0.436,0.583,0.344,24,53,4,22
55,0.60,103,0.252,0.572,0.289,0.846,0.431,0.583,0.344,23,54,4,22
43,0.48,103,0.252,0.552,0.274,1.000,0.430,0.583,0.344,8,69,0,26
54,0.59,103,0.252,0.559,0.282,0.846,0.423,0.583,0.344,21,56,4,22
42,0.47,103,0.252,0.539,0.268,1.000,0.423,0.583,0.344,6,71,0,26
47,0.52,103,0.252,0.546,0.273,0.923,0.421,0.583,0.344,13,64,2,24
41,0.46,103,0.252,0.532,0.265,1.000,0.419,0.583,0.344,5,72,0,26
53,0.58,103,0.252,0.553,0.278,0.846,0.419,0.583,0.344,20,57,4,22


Validation-selected threshold: 0.48999999999999994


,rows,positive_rate,balanced_accuracy,precision,recall,f1,roc_auc,average_precision,TN,FP,FN,TP,threshold
0,62,0.097,0.47,0.091,0.833,0.164,0.479,0.131,6,50,1,5,0.49


## Interpretation

The current experiment does **not** show that the machine-learning regressors outperform the previous-period baseline. The warning classifier also has too many false alarms and a ROC-AUC near/below 0.5 on the later test period, which is inadequate for actual price-shock decisions.

The test contains only six >5% rises, so classification metrics have high uncertainty. Do not adjust the model or threshold based on this test and then report it as an untouched test; this window has now been examined. For the next iteration, use validation/rolling-origin data to choose the model and wait for a later WFP data release (or obtain a verified independent source) for a fresh final test.

### Source notes
- HDX HAPI Food Prices & Market Monitor (WFP series): https://hdx-hapi.readthedocs.io/en/latest/data_usage_guides/food_security_nutrition_and_poverty/
- HAPI price flags/types: https://hdx-hapi.readthedocs.io/en/latest/data_usage_guides/enums/
- World Bank RTFP methodology background: https://documents1.worldbank.org/curated/en/099430004232434765/pdf/IDU-8a5b9997-4299-4e50-ac10-b25f11061d21.pdf

In [8]:
# Package the generated result CSVs into a portable ZIP.
import zipfile, os
output_files = [
    "wfp_worldbank_historical_pairs.csv",
    "wfp_regression_validation_metrics.csv",
    "wfp_regression_future_test_metrics.csv",
    "wfp_regression_future_test_predictions.csv",
    "wfp_classifier_validation_thresholds.csv",
    "wfp_classifier_future_test_metrics.csv",
    "wfp_classifier_future_test_predictions.csv",
]
output_zip = "Yemen_MarketShield_WFP_Validation_Outputs.zip"
with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for filename in output_files:
        path = results_dir / filename
        if path.exists():
            z.write(path, arcname=filename)
print(f"Created {output_zip} with {len([f for f in output_files if (results_dir / f).exists()])} result files.")
if files is not None:
    files.download(output_zip)
else:
    print(f"Local results are in: {results_dir.resolve()}")


Created Yemen_MarketShield_WFP_Validation_Outputs.zip with 7 result files.
Local results are in: /mnt/data/Yemen-MarketShield-AI-Complete-Submission/results
